# 🧠 **GetLit: Your Personal Digital Literacy Coach**

Welcome! This notebook will walk you through three stages:

1. **Meeting your coach:** A short conversation to get to know you and your current level of tech literacy
2. **Taking a quick benchmark quiz:** 6 questions to see where you excel and where you can grow
3. **Getting your plan:** A personalized set of lessons built just for your needs

**How to use this notebook:**
> Run each cell in order by clicking the ▶ button on the left, or press 'Shift + Enter' on your keyboard.
> No need to understand any of the code you see, just follow the prompts!

---
## **Setup**
*Run the cells below once to get everything ready. This will be quick, but is a necessary step for the rest of the process!*

In [1]:
# @title Step 1 of 3: Install required packages (run once)
# @markdown Click ▶ to install. You'll see a checkmark when it's done.
# SET-UP: Installing Google GenAI
!pip install -q google-genai

In [2]:
# @title Step 2 of 3: Connect your API key (run once)
# @markdown This loads your Gemini API key from Colab Secrets. Make sure you've added it as `GEMINI_API_KEY`.
# @markdown If you haven't done that yet: click the key icon in the left sidebar → New secret → Name: GEMINI_API_KEY
# SET-UP: Importing API key
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

In [ ]:
# @title Step 3 of 3: Test the connection (run once)
# @markdown Run this to confirm your API key is working. You should see a greeting from the coach.
# SET-UP: Test call
response = client.models.generate_content(
    model='gemini-pro-latest',
    contents="Say hello and tell me you're ready to help build a digital literacy curriculum."
)
print(response.text)

In [1]:
# @title Load data structures and utilities (run once)
# @markdown Sets up the internal data shapes the pipeline uses. Nothing to see here!
# SET-UP: Defining main data structures

# What Stage 1 will produce
student_profile = {
    "name": "",
    "grade": "",
    "interests": [],
    "self_reported_comfort": "",
    "inferred_comfort": "",
    "goals": ""
}

# What Stage 2 will produce
skill_matrix = {
    "online_safety":              {"score": 0, "note": ""},
    "media_literacy":             {"score": 0, "note": ""},
    "responsible_ai_use":         {"score": 0, "note": ""},
    "file_device_management":     {"score": 0, "note": ""}
}

# What a single lesson will look like
lesson = {
    "title": "",
    "area": "",
    "learning_objective": "",
    "summary": "",
    "interest_hook": "",
    "activity": "",
    "estimated_duration": ""
}

# SET-UP: Utility functions
import json
import re

def extract_json(text):
    # Find JSON block in the response even if there's surrounding text
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        return json.loads(match.group())
    else:
        raise ValueError("No JSON found in response")

---
## **Stage 1:** Meet Your Coach

Your coach will ask you a few questions to get to know you — your name, grade, interests, and how comfortable you feel with technology. Just answer naturally, like you're texting a friend.

**When the conversation is complete, you'll see "Profile complete" and your profile printed below.**

> ▶ Run the cell below to start chatting.

In [8]:
# @title Start your conversation with the coach
# @markdown Type your answers in the input box that appears below each coach message. Press Enter to submit.
# STAGE 1: Getting to know the student

conversation_history = []

system_prompt = """
You are a friendly digital literacy coach having a relaxed first conversation with a student.
Your goal is to genuinely get to know them — not just collect answers, but understand who they are.

Ask ONE question at a time. If a student says something interesting, it's okay to briefly react
before moving on. Keep your tone warm, casual, and encouraging — like a cool tutor, not a survey.

Work through these topics (in order, but naturally):
1. Their name and what grade they're in
2. What they're into — hobbies, interests, things they do for fun
3. Their relationship with technology — ask something like "When you run into a tech problem,
   what do you usually do?" to get a real sense of comfort level
4. Something specific they wish they could do better with technology

Once you have covered all four topics, output ONLY this JSON and nothing else:
{
    "name": "",
    "grade": "",
    "interests": [],
    "self_reported_comfort": "",
    "inferred_comfort": "",
    "goals": ""
}
"""

print("=== GetLit: Digital Literacy Coach ===\n")

while True:
    # Build the message list for this turn
    messages = [{"role": "user" if i % 2 == 0 else "assistant", "content": msg}
                for i, msg in enumerate(conversation_history)]

    # Get the coach's next message
    response = client.models.generate_content(
        model='gemini-pro-latest',
        contents=system_prompt + "\n\n" + "\n".join(
            f"{'Student' if i % 2 == 0 else 'Coach'}: {msg}"
            for i, msg in enumerate(conversation_history)
        ) + ("\nCoach:" if conversation_history else "")
    )

    coach_message = response.text.strip()

    # Check if the model has returned the JSON profile
    try:
        student_profile = extract_json(coach_message)
        print("\n--- Profile complete ---")
        print(json.dumps(student_profile, indent=2))
        break
    except ValueError:
        # Not JSON yet — still in conversation
        print(f"Coach: {coach_message}\n")
        student_input = input("You: ").strip()
        conversation_history.append(coach_message)
        conversation_history.append(student_input)

=== GetLit: Digital Literacy Coach ===

Coach: Hey there! I'm so glad we get to chat today. I'm your digital literacy coach—think of me as a guide to help you get the most out of the tech you use every day, minus all the boring lectures. 

To start us off and get to know you a bit, what's your name and what grade are you in?

You: Grace
Coach: Nice to meet you, Grace! What grade are you in this year?

You: I'm a junior in college
Coach: A junior in college! That is a super exciting (and usually pretty busy!) time. 

When you're not buried in coursework or running to class, what do you like to do for fun? Any hobbies or interests you're really into right now?

You: I really like hiking and cooking
Coach: Hiking and cooking sound like the perfect combo to decompress from classes! There's nothing quite like a good homemade meal after a long day on the trails. 

Since we're going to be talking about tech, I'd love to know a bit about your relationship with it. When you run into a tech prob

---
## **Stage 2:** Take the Quiz

Based on what you shared, the coach will now build a short quiz — 6 questions across four areas of digital literacy. Don't worry about getting everything right! The quiz helps figure out *where to start*, not grade you.

**Run the cells below in order.**

In [9]:
# @title Generate your personalized quiz questions
# @markdown This may take a few seconds. When it's done, you'll see the quiz questions printed below (for reference).
# STAGE 2: Generate quiz questions from student profile

quiz_prompt = f"""
You are a digital literacy assessment tool designing a short quiz for a specific student.
Your goal is to accurately assess their real skill level — not trick them, but genuinely
surface what they know and don't know across four areas of digital literacy.

Student profile:
{json.dumps(student_profile, indent=2)}

Generate exactly 6 multiple choice questions following these rules:

DISTRIBUTION (strictly enforce this):
- 2 questions on online safety
- 2 questions on media literacy
- 1 question on responsible AI use
- 1 question on file and device management

QUESTION QUALITY:
- Each question must have one clearly correct answer and three plausible wrong answers
  (avoid obviously silly distractors)
- Difficulty should match inferred_comfort: "{student_profile['inferred_comfort']}"
  not self_reported_comfort
- At least 2 questions should use a scenario connected to the student's interests:
  {json.dumps(student_profile['interests'])}
- Questions should feel relevant and real, not textbook

AREA VALUES must be exactly one of:
  "online_safety" | "media_literacy" | "responsible_ai_use" | "file_device_management"

Return ONLY a JSON object in this exact format, with no extra text or markdown:
{{
    "questions": [
        {{
            "id": "Q1",
            "area": "",
            "question": "",
            "options": ["A. ", "B. ", "C. ", "D. "],
            "correct_answer": "A"
        }},
        {{
            "id": "Q2",
            "area": "",
            "question": "",
            "options": ["A. ", "B. ", "C. ", "D. "],
            "correct_answer": "A"
        }},
        {{
            "id": "Q3",
            "area": "",
            "question": "",
            "options": ["A. ", "B. ", "C. ", "D. "],
            "correct_answer": "A"
        }},
        {{
            "id": "Q4",
            "area": "",
            "question": "",
            "options": ["A. ", "B. ", "C. ", "D. "],
            "correct_answer": "A"
        }},
        {{
            "id": "Q5",
            "area": "",
            "question": "",
            "options": ["A. ", "B. ", "C. ", "D. "],
            "correct_answer": "A"
        }},
        {{
            "id": "Q6",
            "area": "",
            "question": "",
            "options": ["A. ", "B. ", "C. ", "D. "],
            "correct_answer": "A"
        }}
    ]
}}
"""

quiz_response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=quiz_prompt
)
quiz_data = extract_json(quiz_response.text)
print(json.dumps(quiz_data, indent=2))

{
  "questions": [
    {
      "id": "Q1",
      "area": "online_safety",
      "question": "Grace finds an online store selling high-end hiking gear at a massive 80% discount, but the URL looks slightly unfamiliar. What is the most reliable way to verify the site is safe before entering her payment information?",
      "options": [
        "A. Test the site by entering fake credit card details first to see if it processes an error.",
        "B. Check if the URL starts with HTTPS and search for independent reviews of the specific domain name.",
        "C. Look for a padlock icon anywhere on the webpage and ensure they accept major credit cards.",
        "D. Rely on her browser's built-in ad blocker to automatically block the site if it is malicious."
      ],
      "correct_answer": "B"
    },
    {
      "id": "Q2",
      "area": "online_safety",
      "question": "While studying at a coffee shop on their public Wi-Fi, Grace tries to log into her college portal but gets a browser w

> ▶ Run the cell below to answer your quiz. Type **A**, **B**, **C**, or **D** for each question and press Enter.

In [11]:
# @title Answer your quiz
# @markdown Read each question carefully and type A, B, C, or D. Press Enter after each answer.
# STAGE 2: Collecting student answers

print("=== Quiz Time! ===\n")
print("Answer each question by typing A, B, C, or D.\n")

student_answers = {}

for q in quiz_data["questions"]:
    print(f"{q['id']}: {q['question']}")
    for option in q["options"]:
        print(f"  {option}")
    while True:
        answer = input("Your answer: ").strip().upper()
        if answer in ["A", "B", "C", "D"]:
            student_answers[q["id"]] = answer
            break
        else:
            print("Please enter A, B, C, or D.")
    print()

print("Quiz complete!\n")

=== Quiz Time! ===

Answer each question by typing A, B, C, or D.

Q1: Grace finds an online store selling high-end hiking gear at a massive 80% discount, but the URL looks slightly unfamiliar. What is the most reliable way to verify the site is safe before entering her payment information?
  A. Test the site by entering fake credit card details first to see if it processes an error.
  B. Check if the URL starts with HTTPS and search for independent reviews of the specific domain name.
  C. Look for a padlock icon anywhere on the webpage and ensure they accept major credit cards.
  D. Rely on her browser's built-in ad blocker to automatically block the site if it is malicious.
Your answer: B

Q2: While studying at a coffee shop on their public Wi-Fi, Grace tries to log into her college portal but gets a browser warning that her connection is 'not private.' What is the safest immediate action?
  A. Ignore the warning if the website URL begins with HTTPS, as the connection is already enc

In [12]:
# @title Score your quiz and build your skill snapshot
# @markdown Run this after finishing the quiz. You'll see a summary of how you did.
# STAGE 2: Scoring quiz and building skill matrix (deterministic scorer)

# Re-initialize skill_matrix to its original structure
skill_matrix = {
    "online_safety":              {"score": 0, "note": ""},
    "media_literacy":             {"score": 0, "note": ""},
    "responsible_ai_use":         {"score": 0, "note": ""},
    "file_device_management":     {"score": 0, "note": ""}
}

# Tally correct answers per area
area_totals = {area: {"correct": 0, "total": 0} for area in skill_matrix}

for q in quiz_data["questions"]:
    area = q["area"]
    if area not in area_totals:
        continue  # guard against unexpected area values from the model
    area_totals[area]["total"] += 1
    if student_answers.get(q["id"]) == q["correct_answer"]:
        area_totals[area]["correct"] += 1

# Write scores and diagnostic notes into skill_matrix
def score_note(correct, total):
    ratio = correct / total if total > 0 else 0
    if ratio == 1.0:
        return "strong — got everything right here"
    elif ratio >= 0.5:
        return "partial — some gaps worth addressing"
    else:
        return "needs work — missed most questions in this area"

for area, counts in area_totals.items():
    skill_matrix[area]["score"] = counts["correct"]
    skill_matrix[area]["note"]  = score_note(counts["correct"], counts["total"])

# STAGE 2: Computing overall level and summary

total_correct   = sum(v["correct"] for v in area_totals.values())
total_questions = sum(v["total"]   for v in area_totals.values())
ratio = total_correct / total_questions if total_questions > 0 else 0

if ratio >= 0.75:
    overall_level = "advanced"
elif ratio >= 0.4:
    overall_level = "intermediate"
else:
    overall_level = "beginner"

# Interpreting the matrix and delivering it in instructive language
summary_prompt = f"""
You are summarizing the student's digital literacy quiz results in 2–3 warm, plain-English sentences.
Write directly to the student. Don't use letter or numerical grades.
Note what they're good at but importantly highlight where they have room to grow and where the best place to start is, given their current literacy.


Student profile:
{json.dumps(student_profile, indent=2)}

Skill matrix results:
{json.dumps(skill_matrix, indent=2)}

Overall level: {overall_level}

Return only the summary text, no extra formatting.
"""

summary_response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=summary_prompt
)
summary_text = summary_response.text.strip()

# Putting everything together
scored_results = {
    "skill_matrix":  skill_matrix,
    "overall_level": overall_level,
    "summary":       summary_text
}

print(json.dumps(skill_matrix, indent=2))
print(f"\nOverall level: {overall_level}")
print(f"\nSummary: {summary_text}")

{
  "online_safety": {
    "score": 2,
    "note": "strong \u2014 got everything right here"
  },
  "media_literacy": {
    "score": 2,
    "note": "strong \u2014 got everything right here"
  },
  "responsible_ai_use": {
    "score": 1,
    "note": "strong \u2014 got everything right here"
  },
  "file_device_management": {
    "score": 1,
    "note": "strong \u2014 got everything right here"
  }
}

Overall level: advanced

Summary: Grace, your digital literacy quiz results are excellent – you've shown a strong understanding in every area, from online safety to managing your devices and using AI responsibly. This solid foundation means you're perfectly positioned to dive into your goal of getting better at coding, and exploring introductory programming concepts would be a fantastic next step.


---
## **Stage 3:** Get Your Personalized Learning Plan

Now the coach will build a custom set of lessons for you — one for each area of digital literacy, starting with the areas where you have the most room to grow.

**Run the cells below in order.**

In [13]:
# @title Rank your skill areas by priority
# @markdown This figures out which lessons to build first based on your quiz results.
# STAGE 3: Sorting areas by priority
 # Build a prioritized list of areas from the skill matrix where areas with lower scores come first

area_scores = {
    area: data["score"]
    for area, data in skill_matrix.items()
}

# Sort ascending — lowest score = highest priority
prioritized_areas = sorted(area_scores, key=lambda a: area_scores[a])

print("Lesson priority order:")
for i, area in enumerate(prioritized_areas, 1):
    score = area_scores[area]
    note  = skill_matrix[area]["note"]
    print(f"  {i}. {area} (score: {score}) — {note}")

Lesson priority order:
  1. responsible_ai_use (score: 1) — strong — got everything right here
  2. file_device_management (score: 1) — strong — got everything right here
  3. online_safety (score: 2) — strong — got everything right here
  4. media_literacy (score: 2) — strong — got everything right here


In [14]:
# @title Generate your personalized lessons
# @markdown This builds one lesson per skill area. It may take 10–20 seconds. You'll see a checkmark as each lesson is created.
# STAGE 3: Generating one lesson per skill area starting with the weakest

lesson_generation_prompt = """
You are designing one personalized digital literacy lesson for a specific student. Make it genuinely
engaging for them — not a generic template.

Student profile: {student_profile}
Skill matrix: {skill_matrix}
Overall level: {overall_level}
Target area: "{area}" — {area_note}

Requirements:
- Calibrate difficulty to "{overall_level}"
- Open with a hook tied to their interests: {interests}
- Activity must be concrete and completable in one sitting (not "research and discuss")

Return ONLY a JSON object, no extra text or markdown:
{{
    "title": "",
    "area": "{area}",
    "learning_objective": "Students will be able to...",
    "summary": "",
    "interest_hook": "",
    "activity": "",
    "estimated_duration": ""
}}
"""

curriculum = []  # list of lesson dicts, in priority order

for area in prioritized_areas:
    prompt = lesson_generation_prompt.format(
        student_profile = json.dumps(student_profile, indent=2),
        skill_matrix    = json.dumps(skill_matrix, indent=2),
        overall_level   = overall_level,
        area            = area,
        area_note       = skill_matrix[area]["note"],
        interests       = json.dumps(student_profile["interests"])
    )

    lesson_response = client.models.generate_content(
        model='gemini-pro-latest',
        contents=prompt
    )

    lesson = extract_json(lesson_response.text)
    curriculum.append(lesson)
    print(f"✓ Generated lesson for: {area}")

print(f"\nTotal lessons generated: {len(curriculum)}")

✓ Generated lesson for: responsible_ai_use
✓ Generated lesson for: file_device_management
✓ Generated lesson for: online_safety
✓ Generated lesson for: media_literacy

Total lessons generated: 4


In [15]:
# @title Save full curriculum as JSON
# @markdown This prints the raw data behind your plan. It's mainly for the instructor's sake, so no need to pay too much attention to it!
# STAGE 3: Storing full curriculum as JSON

curriculum_output = {
    "student_name":    student_profile["name"],
    "overall_level":   overall_level,
    "priority_order":  prioritized_areas,
    "lessons":         curriculum
}

print(json.dumps(curriculum_output, indent=2))

{
  "student_name": "Grace",
  "overall_level": "advanced",
  "priority_order": [
    "responsible_ai_use",
    "file_device_management",
    "online_safety",
    "media_literacy"
  ],
  "lessons": [
    {
      "title": "AI Code Auditing: Safe Trail Mapping",
      "area": "responsible_ai_use",
      "learning_objective": "Students will be able to audit AI-generated code for security vulnerabilities and hallucinated functions, demonstrating responsible use of AI tools in software development.",
      "summary": "This lesson bridges Grace's goal of improving at coding with responsible AI use. She will act as a code reviewer for an AI assistant, auditing an AI-generated Python script designed for mapping hiking trails to identify and fix security flaws and hallucinations.",
      "interest_hook": "Whether you're prepping for a multi-day hike or trying out a complex new recipe, you wouldn't blindly trust a random stranger's instructions without double-checking the map or the ingredients.

> ▶ Run the cell below to see your complete digital literacy plan!

In [2]:
# @title Your Digital Literacy Plan
# @markdown Here's your personalized learning plan. Read through your lessons and bookmark this notebook to come back to it!
# STAGE 3: Readable printout

AREA_LABELS = {
    "online_safety":          "Online Safety",
    "media_literacy":         "Media Literacy",
    "responsible_ai_use":     "Responsible AI Use",
    "file_device_management": "File & Device Management"
}

print("=" * 64)
print(f"  DIGITAL LITERACY PLAN — {student_profile['name'].upper()}")
print(f"  Overall Level: {overall_level.capitalize()}")
print("=" * 64)
print(f"\n  {scored_results['summary']}\n")

for i, lesson in enumerate(curriculum, 1):
    label = AREA_LABELS.get(lesson["area"], lesson["area"])
    print(f"  LESSON {i} OF {len(curriculum)}  ·  {label}")
    print(f"  {lesson['title']}")
    print(f"  {lesson['estimated_duration']}")
    print()
    print(f"  Goal")
    print(f"  {lesson['learning_objective']}")
    print()
    print(f"  Overview")
    print(f"  {lesson['summary']}")
    print()
    print(f"  Opening Hook")
    print(f"  {lesson['interest_hook']}")
    print()
    print(f"  Activity")
    for line in lesson["activity"].strip().split("\n"):
        if line.strip():
            print(f"  {line}")
    if i < len(curriculum):
        print(f"\n{'─' * 64}\n")

print(f"\n{'=' * 64}")

NameError: name 'student_profile' is not defined